# Benchmark Quantum Fourier Transform

The Quantum Fourier Transform (QFT) function is a quantum algorithmic primitive, utilized algorithms such as Shor's Factoring, Discrete Logarithm  and Quantum Phase Estimation. It is the quantum analog for
discrete Fourier transform, applied on the quantum variable state vector.

The basis states $|x\rangle_n$ on $n$ qubits, where $x\in\{0,\dots, N-1\}$ and $N=2^n$, transform as

$${\cal{F}}_N|x\rangle = \frac{1}{\sqrt{N}} \sum_{x=0}^{N-1} e^{2\pi i x k/N}| k\rangle ~~.$$

Therefore, for a general state $|\psi\rangle = \sum_{x=0}^{N-1}a_x |x\rangle$,  the components of $|\chi\rangle  = {\cal{F}_N}|\psi \rangle= \sum_{k=0}^{N-1}b_k |k \rangle$.
satisfy $$b_k =\frac{1}{\sqrt{N}} \sum_{x=0}^{N-1}a_x e^{2\pi i x k/N}  $$

***

### The model

We evaluate the QFT of a comb distribution 

$$|\psi_0 \rangle = \frac{1}{\sqrt{2^{N-m}}}\sum_{t=0}^{2^{N-m}-1}|t 2^m\rangle~~, $$

 and compare the outcome population to the expected one


$$ |\psi_f \rangle = \frac{1}{\sqrt{2^{m}}}\sum_{t=0}^{2^m-1}|t 2^{n-m} \rangle~~.$$

We take the simple case of $m=0$, which results in the initial state

$$ |\psi_0 \rangle = \frac{1}{\sqrt{2^{n}}}\sum_{t=0}^{2^{n}-1}|t \rangle~~, $$

which in the qubit representation corresponds to 


$$|\psi_0\rangle =(|0\rangle + |1\rangle)\otimes(|0\rangle + |1\rangle)\otimes \cdots \otimes (|0\rangle + |1\rangle)  = H^{\otimes n} |0^n \rangle~~,$$

and the final state is given by 

$$|\psi_f \rangle = |0^n \rangle $$

The transformation is achieved by Classiq's built-in function `qft`, taking an argument `target`: `QArray[QBit]` a QFT is applied to `target`.


### The Scoring function
For a general comb state we define the score in terms of the **total variation distance** $D_{TV}(p,u) = \frac{1}{2}\sum_{i=0}^{N-1}\left|p_i-u_i \right|$, where $p_i$ and $u_i$ are the observed and expected probabilities, correspondingly. For the comb state we have $u_{t 2^m} = 1/m$ for $t=[0,2^{m}-1]$ and zero for the rest of the probabilities. The distance is bounded between zero and $1-1/n$. To obtain a score between zero and one (one is a perfect result) we define the score as
$$S =1 - \frac{D_{TV}(p,u)}{1-1/n} ~~.$$

***
***

In [1]:
import asyncio
import datetime
import sys
from pathlib import Path

sys.path.insert(0, "..")

import numpy as np
from benchmark import BenchmarkExample
from collector import ResultCollector
from hardware import HardwareRunner
from hardwares_preferences import HARDWARES, print_all_hardwares

from classiq import *


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/tomergoldfriend/user_env3.11/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/Users/tomergoldfriend/user_env3.11/lib/python3.11/site-packages/traitlets/config/application.py", line 1053, in launch_instance
    app.start()
  File "/Users/tomergoldfriend/user_env3.11/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 736, in start
    s

AttributeError: _ARRAY_API not found

In [2]:
# ============================================================
# Fresh start: reset all generated report/results files
# Uncomment to start a new benchmarking project from scratch
#
# from project_reset import reset_benchmark_project
# reset_benchmark_project()
# ============================================================

In [3]:
from examples.qft import QFTExample

***
***
## Benchmarking a 4-qubits QFT

Define a specific example on 4 qubits:

In [4]:
example = QFTExample(problem_size=4, m=0)
example.show()

Quantum program link: https://platform.classiq.io/circuit/3DLICL77ka3N6mNtwK4sQfWWg1a


### Set a `ResultCollector` with a file name fixed for the current example

In [5]:
FILENAME = example.default_results_filename

In [6]:
collector = ResultCollector(FILENAME, build_each_time=True)

In [7]:
# Uncomment to clear the data of a previous run
#
# await collector.reset_file()

### Choose on which backend to run 

We can print the list of backends:

In [8]:
print_all_hardwares(HARDWARES)

IBM Quantum: ibm_kingston, ibm_boston, ibm_marrakesh, ibm_torino, ibm_fez, ibm_pittsburg
IonQ: qpu.forte-1, qpu.forte-enterprise-1, qpu.forte-enterprise-2
Classiq: simulator, simulator_statevector, simulator_density_matrix, nvidia_simulator
Amazon Braket: Ankaa-3, Garnet, Forte 1
Azure Quantum: ionq.qpu.forte-enterprise-1, ionq.qpu.aria-1, ionq.qpu.forte-1


Here we choose one machine for IBM, one for IonQ, as well as Classiq simulators for reference.

*(The list of runners can be edited and added to the benchmarking run via the `ResultCollector`.)*

In [9]:
benchmark_hardware = [
    {"provider": "Classiq", "backend": "simulator"},
    {"provider": "Classiq", "backend": "simulator_statevector"},
    # {"provider": "IonQ", "backend": "qpu.forte-1"},
    # {
    #     "provider": "IBM Quantum",
    #     "backend": "ibm_kingston",
    #     "backend_kwargs": {
    #         "access_token": "PUT YOUR TOKEN HERE",
    #         "channel": "PUT YOUR CHANNEL HERE",
    #         "instance_crn": "PUT YOUR INSTANCE_CRN HERE",
    #     },
    # },
]

Define `HardwareRunner` for each backend, together with the number of shots and other hyperparameters:

In [10]:
common_config = {
    "max_timeout": 5e5,  # value is in seconds. Equals a bit more than 5 days."
    "num_shots": 1000,
}

In [11]:
runners = [
    HardwareRunner(
        cfg["provider"],
        cfg["backend"],
        **common_config,
        **(
            {"backend_kwargs": cfg["backend_kwargs"]} if "backend_kwargs" in cfg else {}
        ),
    )
    for cfg in benchmark_hardware
]

### Run Benchmark

In [12]:
print(
    "=" * 10
    + f"  {example.name}-{example.problem_size} ({datetime.datetime.now()})   "
    + "=" * 10
)
await asyncio.gather(*[collector.run(runner, example) for runner in runners]);

==========  qft-4 (2026-05-06 11:06:35.765250)   ==========
2026-05-06 11:06:38.280413: Submit qft-4 for Classiq - simulator
2026-05-06 11:06:41.018338: Completed qft-4 for Classiq - simulator --> Score {'score': '1.000', 'execution_time': '0.013', 'circuit_depth': 23, 'circuit_width': 4, 'two_qubit_gate_count': 18}
2026-05-06 11:06:41.679640: Submit qft-4 for Classiq - simulator_statevector
** Report updated: qft-4 for Classiq - simulator
2026-05-06 11:06:43.542935: Completed qft-4 for Classiq - simulator_statevector --> Score {'score': '1.000', 'execution_time': '0.017', 'circuit_depth': 24, 'circuit_width': 4, 'two_qubit_gate_count': 18}
** Report updated: qft-4 for Classiq - simulator_statevector


In [13]:
await collector.print_status()

========== (2026-05-06 11:06:44.515175)   ==========
qft-4 | Classiq - simulator | COMPLETED | score=1.000 | time=0.013 min
qft-4 | Classiq - simulator_statevector | COMPLETED | score=1.000 | time=0.017 min
